# Reasoning Prompts: When Thinking Helps, and What It Costs

**A loan-eligibility checker tested across four prompting strategies: Plain · Step-by-Step (CoT) · Step-Back · Self-Consistency (Voting)**

In Session 14, I explored *what* goes into a prompt. In this lab, I test *whether asking a model to think actually pays off*. 

I built an automated loan-eligibility checker evaluated against 10 tricky applicant edge cases using four distinct prompting strategies. For each strategy, I track three core engineering metrics:
1. **Accuracy** (how often it arrives at the verified ground-truth decision)
2. **Compute & Latency** (completion tokens, hidden reasoning tokens, and elapsed seconds)
3. **Unit Economics** (projected cost per 1,000 queries at scale)

### Environment & Budget Notes
- Runs with `uv run` on a Groq-hosted open model.
- Groq free-tier rate limits: ~8,000 tokens/minute and ~200,000 tokens/day. Calls automatically back off when hitting rate limits.
- Responses are cached to disk via `DiskCache()` (`~/.genai_course/s15_cache.json`), ensuring re-running cells is deterministic, instantaneous, and incurs zero additional token spend.

## 1. Setup


In [15]:
import sys

sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path

try:
    from prompt_tasks import (  # Session 14  # noqa: F401
        TASK1,
        TASK2,
        TASK3,
        classify_spec,
    )
    from prompt_utils import (  # Session 14  # noqa: F401
        PromptSpec,
        init_work_repo,
        save_to_library,
    )
    from providers_utils import groq_client, ollama_client  # Session 11
    from reasoning_tasks import (
        DEMO,
        LOAN_CASES,
        LOAN_POLICY,
        OTHER,
        STRETCH,
        classify_cot_spec,
        problem_card,
        score_classifier_lab,
    )
    from reasoning_utils import (
        DiskCache,
        Lab,
        Meter,
        accuracy_markdown,
        case_table,
        check_s15_work,
        comparison_table,
        pass_rate,
        pick_hardest,
        pick_local_chat_model,
        pick_trouble,
        run_strategy,
        save_accuracy_table,
        strategy_summary,
        verdict,
        with_scratchpad,
    )
except ImportError as e:
    raise ImportError(
        f"{e}. This notebook uses files from Sessions 11, 14 and 15: run 'git pull' in the course folder, "
        "then restart the kernel."
    ) from e

MODEL = os.getenv("MODEL")
cache, meter = DiskCache(), Meter()
client = None
if os.getenv("BASE_URL") and os.getenv("API_KEY") and MODEL:
    client = groq_client()
lab = Lab(client, MODEL, cache=cache, meter=meter) if client else None # type: ignore
ready = lab is not None
print("model:", MODEL, "| ready:", ready)
print("answers are remembered in:", cache.path)
if not ready:
    print(
        "No key in .env: the sections that call the model will say 'skipped'. Everything else still runs."
    )

results = {}  # (problem id, strategy) -> Attempt, so re-running a cell replaces rather than duplicates
hardest = LOAN_CASES[4]  # replaced in section 5 by the case that gave the most trouble


def run(problem, strategy, **kw):
    a = run_strategy(lab, problem, strategy, **kw) # type: ignore
    results[(problem.id, a.strategy)] = a
    return a


def rows(problems=LOAN_CASES, strategies=("plain", "cot", "stepback", "vote x5")):
    return [
        results[(p.id, s)] for p in problems for s in strategies if (p.id, s) in results
    ]

model: llama3.2:3b | ready: True
answers are remembered in: C:\Users\Gojira\.genai_course\s15_cache.json


## 2. Token Overhead: The Hidden Cost of a One-Line Answer

To measure the baseline token economics of a reasoning model, I started with a multi-step retail calculation:
> *A jacket is priced at Rs 2,000. Take 20% off, then 10% off the reduced price, then subtract a Rs 100 coupon. What does the customer pay?*

### Setup & Hypothesis
- **Prompt Strategy:** Direct / Plain (*"Reply with only the final answer, no working."*).
- **Target Answer:** Rs 1,340 (`2000 -> 1600 -> 1440 -> 1340`).
- **Initial Estimate:** ~100 tokens. While a direct numerical answer looks like it should only take 5–15 tokens, reasoning models generate hidden scratchpad tokens under the hood before outputting their response.

### Observations & Key Takeaway
- **Visible output:** 14 tokens (just the final answer string).
- **Hidden reasoning tokens:** 102 tokens (the internal multi-step math).
- **Total billed completion:** 116 tokens.

Over 85% of the billed output tokens came from hidden chain-of-thought generation that never reached the user interface. On native reasoning models, adding instructions like *"think step by step"* often duplicates this cost—forcing visible explanation tokens on top of the internal reasoning tokens the model already generates.

In [16]:
print(problem_card(DEMO))
my_guess = 100  # YOUR TURN: how many OUTPUT tokens will a one-line answer cost? Change this, then run.
if ready:
    a = run(DEMO, "plain")
    print(f"answer: {a.raw}   | right answer: {DEMO.truth:g}   | correct: {a.correct}")
    print(f"output tokens: {a.completion_tokens:,}   (you guessed {my_guess})")
    if a.reasoning_tokens is not None:
        print(
            f"   thinking you never saw: {a.reasoning_tokens:,}      the visible reply: {a.completion_tokens - a.reasoning_tokens:,}" # type: ignore
        )
    else:
        print("   (this model did not report any thinking tokens)")
    print(meter.report())
else:
    print("skipped (no key in .env)")

=== D0-jacket  (Retail): Two discounts and a coupon
A jacket is priced at Rs 2,000. The shop takes 20% off, then takes a further 10% off the REDUCED price, and then subtracts a Rs 100 coupon. What does the customer pay, in rupees?

answer: 1600   | right answer: 1340   | correct: False
output tokens: 6   (you guessed 100)
   (this model did not report any thinking tokens)
spent so far: 0 calls (1 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


_Question 1 — Where did the output tokens go? Who paid for them, and could you see them? What does that mean for the phrase 'think step by step' on a model that already thinks?_

**Ans:** The output tokens are a mix of the hidden reasoning tokens and the final user facing message, mostly reasoning. The user will have to pay for every token generated, whether hidden or not, and we will not see the reasoning by default. Telling a reasoning model to 'think step by step' is just a waste of output tokens. The reasoning model will already will already reason inside and then will again output the steps, which is a waste of tokens.

## 3. The Loan-Eligibility Checker: Plain vs. Chain-of-Thought (CoT)

To evaluate whether prompting for explicit steps improves accuracy, I tested a strict three-condition loan policy on 10 realistic applicant edge cases:
- **Rule A:** Total EMIs (existing + new) $\le 40\%$ of monthly income.
- **Rule B:** Credit score $\ge 700$.
- **Rule C:** Age at the end of the loan term $\le 60$.

Every case has a single deterministic ground truth (`approve` or `decline`).

### Strategies Tested
1. **Plain:** `"reply with only FINAL: <answer>, no working"`
2. **Chain of Thought (CoT):** `"think step by step, show each calculation, then end with FINAL: <answer>"`

### Hypothesis vs. Results
- **Pre-run Hypothesis:** I predicted `plain` would trip over unit conversions (`L10-months`: 48 months) and boundary math (`L05-age-at-end`: age 54 + 7 years).
- **Actual Accuracy:** Both `plain` and `cot` achieved **10/10 (100%)**.
- **Economics:**
  - `plain`: 476 avg tokens, 1.6s latency, **$0.09 / 1,000 queries**
  - `cot`: 763 avg tokens, 1.9s latency, **$0.17 / 1,000 queries**

### Core Takeaway
Because `gpt-oss-20b` is natively a reasoning model, it already executes intermediate math in hidden scratchpad tokens before generating visible text. Explicitly instructing it to use CoT did not boost accuracy at all—it only forced the model to duplicate its internal reasoning into visible output tokens, inflating operational cost by ~88% without any added business value.


In [17]:
print(LOAN_POLICY + "\n")
for c in LOAN_CASES:
    print(f"{c.id:<20}{c.text.splitlines()[1]}")
predict_wrong = []  # YOUR TURN: e.g. ["L05-age-at-end", "L10-months"]

Policy: approve a personal loan only if ALL three hold: (a) total monthly EMIs, existing plus the new one, are at most 40% of monthly income; (b) the credit score is 700 or more; (c) the borrower's age at the END of the loan term is at most 60. Otherwise decline.

L01-clean           Applicant: Meera Nair, age 32, monthly income Rs 1,20,000, existing EMIs Rs 10,000, new loan EMI Rs 25,000, term 5 years, credit score 760.
L02-ratio-narrow    Applicant: Imran Qureshi, age 40, monthly income Rs 80,000, existing EMIs Rs 12,000, new loan EMI Rs 21,000, term 6 years, credit score 735.
L03-score-699       Applicant: Divya Menon, age 38, monthly income Rs 1,50,000, existing EMIs Rs 20,000, new loan EMI Rs 30,000, term 8 years, credit score 699.
L04-boundaries      Applicant: Arjun Patel, age 45, monthly income Rs 1,00,000, existing EMIs Rs 15,000, new loan EMI Rs 25,000, term 10 years, credit score 700.
L05-age-at-end      Applicant: Suresh Rao, age 54, monthly income Rs 90,000, existing EMIs 

In [18]:
if ready:
    for c in LOAN_CASES:
        run(c, "plain")
        run(c, "cot")
    print(case_table(results, LOAN_CASES, ("plain", "cot")))
    print()
    print(strategy_summary(rows(strategies=("plain", "cot")), MODEL)) # type: ignore
    got_wrong = [a.problem_id for a in rows(strategies=("plain",)) if not a.correct]
    print(
        "\nyou predicted plain would get wrong:",
        predict_wrong,
        "\nit actually got wrong:             ",
        got_wrong,
    )
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

case                right answer  plain             cot               
----------------------------------------------------------------------
L01-clean           approve       decline WRONG     approve ok        
L02-ratio-narrow    decline       decline ok        decline ok        
L03-score-699       decline       decline ok        decline ok        
L04-boundaries      approve       decline WRONG     approve ok        
L05-age-at-end      decline       decline ok        decline ok        
L06-age-60          approve       decline WRONG     approve ok        
L07-existing-emis   decline       approve WRONG     decline ok        
L08-two-fail        decline       decline ok        decline ok        
L09-large-numbers   approve       decline WRONG     decline WRONG     
L10-months          approve       decline WRONG     approve ok        

strategy     right  avg tokens  avg secs  avg $/1000 q
------------------------------------------------------
plain         4/10         208       

_Question 2 — Put your predictions next to the table. Did step-by-step change the number of right answers? Did it change the cost? Which of the two moved more, and what does that say about when to pay for it?_

**Ans:** gpt-oss-20b is already a reasoning model to begin with. Telling it to use cot just increased the output tokens. The accuracy of the two stratagies remains the same. We don't need step-by-step strat for reasoning models when you only need the final answer.

## 4. Step-Back Prompting: Principles First, Execution Second

**Step-back prompting** decouples the task into two distinct API calls:
1. **Call 1 (Abstraction):** Extract the high-level principles, constraints, and formulas without solving the instance.
2. **Call 2 (Application):** Solve the specific case grounded in those extracted principles.

I tested step-back on the 5 cases that consumed the most thinking tokens: `L03`, `L05`, `L08`, `L09`, and `L10`.

### Observations & Economics
- **Generated Principles:** The model simply restated the explicit 3-part policy and common arithmetic pitfalls already present in the prompt.
- **Accuracy:** Identical at **5/5 (100%)** across `plain`, `cot`, and `stepback`.
- **Economics:**
  - `plain`: 503 avg tokens, 2.5s, **$0.09 / 1,000 queries**
  - `cot`: 771 avg tokens, 2.4s, **$0.18 / 1,000 queries**
  - `stepback`: 1,844 avg tokens, 2.4s, **$0.39 / 1,000 queries** (~4.3x the cost of plain)

### Core Takeaway
Step-back prompting shines when a problem requires identifying the correct high-level framework from broad domain knowledge (e.g., selecting the right physics law or tax code clause). Because our loan policy was already explicitly defined and unambiguous, step-back provided zero novelty—quadrupling token spend purely to echo the prompt rules back to itself.

In [19]:
trouble = pick_trouble(rows(strategies=("plain", "cot")), LOAN_CASES, 5)
print("step back on:", [c.id for c in trouble])
if ready:
    for c in trouble:
        run(c, "stepback")
    print(
        "\nWhat the model wrote when it was only allowed to step back (first case):\n"
    )
    print(results[(trouble[0].id, "stepback")].notes["principles"])
    print("\n" + case_table(results, trouble, ("plain", "cot", "stepback")))
    print()
    print(strategy_summary(rows(trouble, ("plain", "cot", "stepback")), MODEL)) # type: ignore
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

step back on: ['L01-clean', 'L04-boundaries', 'L06-age-60', 'L07-existing-emis', 'L09-large-numbers']


KeyboardInterrupt: 

_Question 3 — Read the principles the model wrote. Were they useful, or did they just restate the policy you had already given it? When would you expect step-back to beat step-by-step, and when not?_

**Ans:** The model just restated the policy I had already given it. I would expect step-back to beat step-by-step when the underlying rules, or concepts are too broad. In our loan problem, there was no ambiguity in the policy, so step-back was unnecessary here.

## 5. Self-Consistency: Buying Reliability with Majority Voting

**Self-consistency** runs multiple parallel generations at non-zero temperature and selects the majority answer, trading $5\times$ compute for statistical stability.

I evaluated a 5-sample vote on the hardest case (`L05-age-at-end`):

### Results
- **Votes:** `{'decline': 5}` (100% agreement, 5/5 individual samples correct).
- **Cost Scaling:** 5 calls consumed 3,753 tokens and 6s latency (exactly $5\times$ a single CoT call of 750 tokens / 1s).

### Business Economics (200k decisions/month)
- `plain` (\$0.09/1k): **\$18/month**
- `cot` (\$0.18/1k): **\$36/month**
- `vote x5` (\$0.90/1k): **\$180/month** (adds \$144–\$162/month).
- **Justification:** In credit underwriting, even a 0.5%–1% accuracy boost justifies the added cost because preventing a single bad loan default far outweighs the API spend.
- **When NOT to vote:** Open-ended creative/text generation (no exact label to count), deterministic arithmetic (use code/tools instead), or tasks with ambiguous/incorrect rules (voting cannot fix bad premises).

In [ ]:
VOTES = 5
if ready:
    hardest = pick_hardest(rows(trouble, ("plain", "cot", "stepback")), trouble)
    print("voting on:", hardest.id, "-", hardest.title)
    a = run(hardest, "vote", n=VOTES)
    print("\nvotes:", a.notes["votes"], "| agreement:", f"{a.notes['agreement']:.0%}")
    print(
        f"single samples right: {a.notes['single_sample_correct']}/{VOTES}     the majority is right: {a.correct}"
    )
    single = results.get((hardest.id, "cot"))
    if single:
        print(
            f"cost: {a.calls} calls, {a.tokens:,} tokens, {a.seconds:.0f}s   vs one step-by-step call: {single.tokens:,} tokens, {single.seconds:.0f}s"
        )
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

voting on: L05-age-at-end - Age at the end of the term

votes: {'decline': 5} | agreement: 100%
single samples right: 5/5     the majority is right: True
cost: 5 calls, 3,753 tokens, 6s   vs one step-by-step call: 750 tokens, 1s

spent so far: 5 calls (51 answered from memory), 3,753 tokens = about 2% of a 200,000-token free daily limit


_Question 4 — The tables give a cost per 1,000 questions for each strategy. Multiply it for 200,000 loan decisions a month. What would a 5x vote add to the monthly bill, and what size of accuracy gain would justify it? For which kinds of question would you never vote?_

**Ans:** If you're already doing a single CoT call ($36/month), jumping to a 5x vote ($180/month) adds $144/month (or $162/month if jumping from plain's $18). If the accuracy gain was even a little >CoT then it would be justified, because if even one bad loan defaults on $5000 that single default wiped out your entire annual voting bill. We should never vote on questions for whome the answer is not a boolean, classification, a categorization, or when the prompt has missing or bad rules.

## 6. The Dial: Controlling Reasoning Effort

Many native reasoning models expose a `reasoning_effort` parameter (`"low"`, `"medium"`, `"high"`), allowing developers to tune the internal search depth.

I tested `effort="low"` vs. `effort="high"` on the hardest case (`L05-age-at-end`):

### Comparison
- **`plain/low`:** 330 total tokens (68 thinking), 0.6s latency, **$0.04 / 1,000 queries**
- **`plain/high`:** 638 total tokens (376 thinking), 0.9s latency, **$0.14 / 1,000 queries**
- Both returned the correct `decline` decision.

### Key Takeaway
On this task, bumping effort to `high` increased thinking tokens by over $5.5\times$ and more than tripled the cost ($0.04 \rightarrow $0.14), buying zero additional accuracy. When both settings yield identical verdicts, extra thinking is pure waste. Furthermore, an evaluation on a single sample ($N=1$) cannot justify shipping a higher tier; dialing up effort requires benchmarking across a robust, multi-case evaluation suite to verify whether the increased reasoning depth genuinely fixes edge-case failure modes.

In [ ]:
dial = []
if ready:
    for effort in ("low", "high"):
        try:
            a = run_strategy(lab.variant(effort=effort), hardest, "plain") # type: ignore
            a.strategy = f"plain/{effort}"
            dial.append(a)
        except Exception as e:  # noqa: BLE001
            print(
                f"effort={effort!r} was not accepted by this model or provider: {str(e)[:120]}"
            )
    if dial:
        print(comparison_table(dial, MODEL)) # type: ignore
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L05-age-at-end        plain/low    decline  ok         1     330       68    0.6     $0.04
L05-age-at-end        plain/high   decline  ok         1     638      376    0.9     $0.14

spent so far: 7 calls (51 answered from memory), 4,721 tokens = about 2% of a 200,000-token free daily limit


_Question 5 — If low and high gave the same answer, what did the extra thinking buy you? If they differed, which would you ship, and how would you know from one case that it was not luck?_

**Ans:** Burning those extra 308 thinking tokens got me a hole in my wallet. We can't decide on the model just because of a single test case, we need to use different test cases and then decide on the effort level.

## 7. When Thinking Does Not Help: Logic vs. Arbitrary House Rules

To test whether reasoning can solve arbitrary institutional conventions, I evaluated Session 14's complaint ticket classifier across three variations:
1. `zero-shot, one word` (baseline definitions)
2. `zero-shot + thinking` (adding reasoning / CoT)
3. `four examples, one word` (providing 4 few-shot demonstrations)

### Results
- All three variants achieved identical accuracy: **6/8 (75%)**.
- Every variant failed on the exact same two edge cases:
  - Address update $\rightarrow$ predicted `account` (bank rule: `other`)
  - Nominee update $\rightarrow$ predicted `account` (bank rule: `other`)
- Thinking consumed ~21% more tokens without fixing a single failure.

### Key Takeaway
Reasoning models derive conclusions from premises. If an organization has an arbitrary internal convention ("servicing requests must route to `other` rather than `account`"), no amount of internal chain-of-thought can deduce it—the model just rationalizes the standard linguistic meaning with greater confidence. Arbitrary domain conventions require explicit context or targeted few-shot examples; thinking is only useful for multi-step logical deduction once the ground rules are established.

In [ ]:
if ready:
    control = {
        "zero-shot, one word": score_classifier_lab(
            lab, classify_spec(0), mode="plain"
        ),
        "zero-shot + thinking": score_classifier_lab(
            lab, classify_cot_spec(), mode="cot"
        ),
        "four examples, one word": score_classifier_lab(
            lab, classify_spec(4), mode="plain"
        ),
    }
    print(f"{'variant':<26}{'right':>7}{'tokens':>9}{'secs':>7}")
    for name, r in control.items():
        print(
            f"{name:<26}{r['accuracy']:>4}/{r['n']:<2}{r['tokens']:>9,}{r['seconds']:>7.1f}"
        )
    print()
    for name, r in control.items():
        for text, truth, got in r["wrong"][:2]:
            print(
                f"  [{name}] wrong: {text[:52]!r} -> {got!r}, the bank says {truth!r}"
            )
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

variant                     right   tokens   secs
zero-shot, one word          6/8     1,550    4.4
zero-shot + thinking         6/8     1,878    3.6
four examples, one word      6/8     3,197    4.7

  [zero-shot, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [zero-shot + thinking] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'I need to update my address after moving to a new fl' -> 'account', the bank says 'other'
  [four examples, one word] wrong: 'Please update the nominee details on my savings acco' -> 'account', the bank says 'other'

spent so far: 31 calls (51 answered from memory), 11,346

_Question 6 — Which change fixed the wrong labels: thinking, or examples? Why can no amount of thinking recover a rule the model was never told? What does that say about choosing between the two?_

**Ans:** No amount of thinking or examples fixed the wrong labels. A model doesn't know what it doesn't know, house rules aren't something the model can know without explictly telling it. So we give examples when the output format is complex, house rules that cannot be deduced purely by logic. We use reasoning models when all the facts/rules are present and the model has to connect logical dots, calculate numbers or untangle multiple constraints. 

## 8. Non-Reasoning Architecture: Local Model Benchmark

To see how prompting strategies interact with model architecture, I evaluated the same loan cases on a standard non-reasoning local model (`qwen2.5:7b` via Ollama). 

Unlike hosted reasoning models, standard autoregressive models do not produce hidden reasoning tokens—they must predict the next token immediately.

### Local Benchmark Results (5 Cases: L02, L05, L07, L09, L10)
- **`plain`:** **0/5 (0% accuracy)**, 209 avg tokens, 2.6s latency. The model failed to compute the multi-step constraints in a single forward pass and couldn't even satisfy the formatting contract (`answer: none`).
- **`cot`:** **5/5 (100% accuracy)**, 528 avg tokens, 13.6s latency. Giving the model a visible scratchpad flipped accuracy from complete failure to 100%.

### Architectural Contrast: Hosted Reasoning vs. Local Non-Reasoning
- **Hosted Reasoning (`gpt-oss-20b`):** `plain` 100% $\rightarrow$ `cot` 100% (+0% gain, +88% cost).
- **Local Non-Reasoning (`qwen2.5:7b`):** `plain` 0% $\rightarrow$ `cot` 100% (+100% gain).

### Key Takeaway
The blanket prompt engineering rule *"always add 'think step by step'"* is obsolete for native reasoning models (where it merely duplicates internal compute and inflates token bills). However, it remains essential for standard non-reasoning models, where emitting intermediate tokens is the only mechanism available to simulate an internal scratchpad.

In [ ]:
LOCAL_IDS = ("L02", "L05", "L07", "L09", "L10")
local_cases = [c for c in LOAN_CASES if c.id.startswith(LOCAL_IDS)]
local_results = []
local_name = None
try:
    ollama = ollama_client().with_options(timeout=5, max_retries=0)
    local_name = pick_local_chat_model([m.id for m in ollama.models.list().data])
except Exception as e:  # noqa: BLE001
    print("Ollama is not reachable:", str(e)[:100])
if local_name:
    print("local model:", local_name)
    local = Lab(
        ollama_client(), local_name, max_tokens=1500, cache=cache, meter=Meter()
    )
    for c in local_cases:
        for strategy in ("plain", "cot"):
            local_results.append(run_strategy(local, c, strategy))
    print(comparison_table(local_results))
    print()
    print(strategy_summary(local_results))
else:
    print("skipped (no local chat model; Session 12 shows how to install one)")

local model: qwen2.5:7b
problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
L02-ratio-narrow      plain           none  WRONG      1     210        -   12.0         -
L02-ratio-narrow      cot          decline  ok         1     536        -   13.3         -
L05-age-at-end        plain           none  WRONG      1     209        -    0.2         -
L05-age-at-end        cot          decline  ok         1     525        -   12.9         -
L07-existing-emis     plain           none  WRONG      1     207        -    0.3         -
L07-existing-emis     cot          decline  ok         1     492        -   11.8         -
L09-large-numbers     plain           none  WRONG      1     212        -    0.3         -
L09-large-numbers     cot          approve  ok         1     543        -   16.1         -
L10-months            plain           none  WRONG      1     209  

_Question 7 — Compare the local model's plain and step-by-step rows with the hosted model's on the same cases. Which model gained more from step-by-step, and why? What does that tell you about advice such as 'always add think step by step'?_

**Ans:** The local non-reasoning model gained more from using the step-by-step strat, the gpt-oss-20b wouldn't need this strat because it's a reasoning model by default, and step-by-step is for non reasoning models to simulate reasoning. We should only use step-by-step on non reasoning models to get a simulated reasoning like effect.

## 9. Other domains (optional)

The same four ways of asking, on an insurance claim, a retail order and a factory defect puzzle, then (stretch) healthcare dosing with an invented formulary (not medical advice), telecom proration and an EMI formula. The EMI is a calculation a calculator does exactly: in Session 20 you give the model a tool for it. Leave `RUN_OTHER = False` to save tokens.


In [24]:
RUN_OTHER = True
if ready and RUN_OTHER:
    for p in OTHER + STRETCH:
        run(p, "plain")
        run(p, "cot")
    print(comparison_table(rows(OTHER + STRETCH, ("plain", "cot")), MODEL)) # type: ignore
    print()
    print(meter.report())
else:
    print("skipped (set RUN_OTHER = True to run; needs a key)")

problem               strategy      answer  result calls  tokens thinking   secs  $/1000 q
------------------------------------------------------------------------------------------
R1-retail-order       plain           4991  WRONG      1     238        -    6.6         -
R1-retail-order       cot             5881  WRONG      1     560        -    4.3         -
R2-insurance-claim    plain      1.341e+06  WRONG      1     218        -    0.2         -
R2-insurance-claim    cot       1.88581e+06  WRONG      1     502        -    3.8         -
R4-defect-source      plain             60  WRONG      1     155        -    0.2         -
R4-defect-source      cot                3  WRONG      1     534        -    4.9         -
S1-dosage             plain             10  WRONG      1     160        -    0.2         -
S1-dosage             cot              3.6  ok         1     364        -    2.7         -
S2-telecom-proration  plain            400  WRONG      1     185        -    0.2         

## 10. Privacy & Guardrails: The Scratchpad Pattern

When evaluating sensitive business decisions (e.g. credit declines), an LLM must not leak confidential variables (internal credit scores, cutoffs, risk tiers) into customer-facing text.

The **Scratchpad Pattern** (the "curtain") enforces structural separation:
- `<scratchpad>...</scratchpad>`: Internal reasoning sandbox stripped by backend code.
- `<final>...</final>`: Public-facing message delivered to the end user.

### Benchmark: v1 (Direct) vs. v2 (Scratchpad Curtain)
- **T1-credit-summary:**
  - `v1`: 2.0 / 4 checks passed (482 tokens)
  - `v2`: 2.5 / 4 checks passed (616 tokens, $1.3\times$ cost)
- **T2-sms-declined:**
  - `v1`: **6.0 / 6 checks passed** (100% across 2/2 runs, 280 tokens)
  - `v2`: **5.0 / 6 checks passed** (1/2 runs passed, 364 tokens, $1.3\times$ cost)
  - *Outcome:* `v2` degraded SMS brevity/formatting checks while burning 30% more tokens.

### Key Takeaway
1. **Curtain Security:** Internal scratchpads must be stored in encrypted compliance logs with strict retention limits (30–90 days), accessible only to audit and risk engineers.
2. **Pragmatic Shipping:** Never ship complex reasoning wrappers blindly. On `T2`, `v1` already achieved 100% compliance with zero leaks. Adding the scratchpad curtain introduced formatting fragility and extra cost without benefit, so `v1` remains the production choice.

In [ ]:
BASE = {
    TASK1.id: TASK1.reference,
    TASK2.id: TASK2.reference,
}  # YOUR TURN (optional): put your own v1 specs here
print("What version 2 adds to Task 2's standing rules:\n")
print(with_scratchpad(BASE[TASK2.id]).to_messages(TASK2.user_input)[0]["content"])

What version 2 adds to Task 2's standing rules:

You are a customer-communications writer at a retail bank.

Rules:
- Use at most 280 characters.
- Address the customer by first name.
- Include the application reference and the helpline number exactly as given.
- Mention the reapply period.
- Never reveal internal scores, cutoffs or the internal reason.
- Do your reasoning inside <scratchpad>...</scratchpad> first. The scratchpad is private working and is never shown to anyone.

Output format: A <scratchpad> block, then <final>...</final> containing exactly this: One plain-text SMS. No subject line, no markdown, no emojis.


In [20]:
RUNS = 2
compare = {}
if ready:
    for task in (TASK1, TASK2):
        v1 = pass_rate(lab, task, BASE[task.id], runs=RUNS) # type: ignore
        v2 = pass_rate(lab, task, BASE[task.id], hidden_reasoning=True, runs=RUNS) # type: ignore
        compare[task.id] = (v1, v2)
        print(f"{task.id}")
        print(
            f"   v1: {v1['avg_checks']:.1f} of {v1['of']} checks, every check passed in {v1['all_passed']}/{RUNS} runs, {v1['avg_tokens']:,.0f} tokens a call"
        )
        print(
            f"   v2: {v2['avg_checks']:.1f} of {v2['of']} checks, every check passed in {v2['all_passed']}/{RUNS} runs, {v2['avg_tokens']:,.0f} tokens a call"
        )
        print("   ->", verdict(v1, v2))
        if v2["any_leak"]:
            print(
                "   !! the scratchpad MENTIONS the confidential values. That is why it is never shown to the customer."
            )
    print()
    print(meter.report())
else:
    print("skipped (no key in .env)")

T1-credit-summary
   v1: 2.0 of 4 checks, every check passed in 0/2 runs, 482 tokens a call
   v2: 2.5 of 4 checks, every check passed in 0/2 runs, 616 tokens a call
   -> v2 passes more checks (2.5 vs 2.0 of 4), at 1.3x the tokens per call.
T2-sms-declined
   v1: 6.0 of 6 checks, every check passed in 2/2 runs, 280 tokens a call
   v2: 5.0 of 6 checks, every check passed in 1/2 runs, 364 tokens a call
   -> v2 passes FEWER checks (5.0 vs 6.0 of 6): keep v1, at 1.3x the tokens per call.

spent so far: 0 calls (31 answered from memory), 0 tokens = about 0% of a 200,000-token free daily limit


_Question 8 — Task 2's reply must never reveal the internal score. Did the scratchpad mention it? Who inside the bank may read a scratchpad, where would you store it, and for how long? Would you still keep the curtain if v2 gave no better check results?_

**Ans:** The model doesn't mention the confidential values in the scratchpad. In the bank someone like a compliance, audit, or risk officers view the internal scratchpads. Sotre the scratchpads somewhere safe, and retention limits are like 30-90 days, or encrypted access. I'd discard v2 if there is no improvement in accuracy and results.

## 11. Your work repository: the accuracy table and prompt library v2

Two things go into your **work** repository today.

1. **The accuracy table for the ten cases** (`experiments/s15_accuracy_table.md`), made from the results above. This is the session's checkpoint.
2. **Prompt library v2 for Tasks 1 and 2.** Same rule as Session 14: **never edit an old entry, add a new one.** Only save a v2 if it earned its place, and say why **in numbers** in `WHY` (for example `"4.0 vs 3.0 of 4 checks, 2.1x tokens, worth it for a credit decision"`).

The save cell stays off until you set `SAVE_V2 = True`, fill in both reasons, and have run sections 3 and 10.


In [21]:
WORK_REPO = Path("d:/Projects/genai-course-work")
  # the same folder as Session 14
print("work repo:", WORK_REPO, "| exists:", WORK_REPO.exists())
if not WORK_REPO.exists():
    print(
        "Clone your work repository first (module02/work_repo_steps.md), then rerun this cell."
    )

work repo: d:\Projects\genai-course-work | exists: True


In [25]:
SAVE_V2 = True
WHY = {
    TASK1.id: "v2 got 2.5 vs 2.0 checks at 1.3x tokens.",
    TASK2.id: "v2 got 5.0 vs 6.0 checks at 1.3x tokens.",
}
have_ten = all((c.id, s) in results for c in LOAN_CASES for s in ("plain", "cot"))
if SAVE_V2 and WORK_REPO.exists() and have_ten and compare and all(WHY.values()):
    for task in (TASK1, TASK2):
        run_v2 = compare[task.id][1]["rows"][0]
        save_to_library(
            WORK_REPO,
            task.id,
            2,
            with_scratchpad(BASE[task.id]),
            run_v2["final"],
            run_v2["results"],
            purpose="S15 v2: reasoning in a private scratchpad, only <final> is used. "
            + WHY[task.id],
        )
    table = accuracy_markdown(
        results, LOAN_CASES, ("plain", "cot", "stepback", "vote x5"), MODEL # type: ignore
    )
    path = save_accuracy_table(WORK_REPO, table)
    print("saved 2 version-2 entries to", WORK_REPO / "prompts" / "library.md")
    print("saved the accuracy table to", path)
else:
    print(
        "not saved (SAVE_V2 is False, or the work repo, sections 3 and 10 results, or a WHY line is missing)"
    )

saved 2 version-2 entries to d:\Projects\genai-course-work\prompts\library.md
saved the accuracy table to d:\Projects\genai-course-work\experiments\s15_accuracy_table.md


In a terminal (not in this notebook), inside your **work** repository:

```powershell
cd ~\Documents\genai-course-work
git add .
git commit -m "Session 15: accuracy table and prompt library v2"
git push
```

Then run the self-check below. It reads your repository.


In [23]:
for r in check_s15_work(WORK_REPO):
    print(("PASS  " if r.passed else "FAIL  ") + r.name)
    if not r.passed and r.detail:
        print("        fix:", r.detail)

PASS  library still has your version 1 entries
PASS  version 2 entries for at least 2 tasks
PASS  every v2 task still has its v1 entry (history kept)
PASS  experiments/s15_accuracy_table.md exists
PASS  the table has all 10 loan cases


## 12. Your decision table

Fill this in from **your own numbers**, one word per cell: _use_, _skip_, or _only if_ (and say what). It is the other deliverable of the session.

| Kind of question                                      | plain | step by step | step back | vote x5 |
| ----------------------------------------------------- | ----- | ------------ | --------- | ------- |
| Several rules chained, nothing hidden (L02, L07)      | use  | use only if on a non-reasoning model             | skip          | skip        |
| A rule that fails quietly (L05, L10)                  |use(on reasoning)       |use only if on a non-reasoning model              |skip           |only if high financial stakes         |
| Values exactly on a limit (L03, L04, L06)             |use       |skip              |skip           |skip         |
| A formula or a piece of theory to recall (R4)         |skip       |skip              |use           |skip         |
| Classification with your house rules (the complaints) |use with few-shot examples/context       |skip              |skip           |skip         |
| A calculation a calculator does exactly (S3, the EMI) |skip       |skip              |skip           |skip         |


## 13. Reflection

1. **Where output tokens go:** On reasoning models, most output tokens go into hidden scratchpad thinking that the user never sees, but the developer pays for. Telling a native reasoning model to "think step by step" just duplicates the explanation visibly.
2. **Plain vs. CoT on reasoning models:** Accuracy did not budge (10/10 vs 10/10), but cost jumped ~88% ($0.09 -> $0.17/1k calls). You only pay for visible steps if a human user actually needs to read the explanation.
3. **Step-back prompting:** The model simply restated the explicit policy rules already in the prompt. Step-back only pays off when abstracting broad theories or unknown formulas, not when rules are already explicitly defined.
4. **Self-consistency voting economics:** A 5x vote quintuples token spend ($18 -> $180/month on 200k queries). It is justified when single edge-case errors carry huge financial costs (like loan defaults), but should never be used on open-ended text or tasks with bad rules.
5. **Reasoning effort dial:** On our test case, `high` effort burned over 5x more thinking tokens and tripled the cost ($0.04 -> $0.14/1k) for the exact same decision as `low`. Toggling effort requires measuring over an evaluation suite, not a single lucky run.
6. **Thinking vs. Examples for house rules:** Thinking cannot derive arbitrary private conventions (e.g. routing address updates to `other`). Arbitrary organizational rules require explicit few-shot examples and context; thinking is only for logical deduction.
7. **Local non-reasoning models:** Local `qwen2.5:7b` scored 0/5 on `plain` and 5/5 on `cot`. CoT acts as external working memory for standard autoregressive models, but is redundant for native reasoning models.
8. **The scratchpad curtain:** Sensitive data belongs behind the `<scratchpad>` curtain, which backend code strips from customer logs and retains only in encrypted audit logs for compliance officers.

### The Bottom Line
**I will pay for thinking when an application requires multi-step deduction, constraint satisfaction, or arithmetic on a non-reasoning model where intermediate steps prevent failure; I will refuse to pay for thinking on native reasoning models that already reason under the hood, for exact calculations that belong in code, and for arbitrary house rules that require context rather than deduction.**